# 特征工程 — 质量评价

> 以下为独立第三方视角的评价，供学习参考。

## 整体评价：入门级特征工程，覆盖了核心逻辑

### 做得好的地方
- 覆盖了全部 5 张子表，没有遗漏数据源
- 特征设计有业务逻辑支撑，不是机械聚合（如 ATM 取现比例、最低还款比例）
- NaN 处理合理，fillna 策略正确
- 发现并修复了除以 0 产生 inf 的问题

### 不足与可改进方向

**1. 特征数量偏少**
- 从 5 张子表共提取约 23 个特征；竞赛银牌方案通常提取数百个特征再通过特征选择筛选
- 可对多列批量使用 `.agg(["mean", "max", "min", "sum", "std"])` 快速扩充

**2. 聚合维度单一**
- 只在 `SK_ID_CURR` 级别做了一层聚合
- 未考虑时间维度（如"最近 6 个月的逾期"vs"全部历史逾期"，近期行为预测力更强）
- 未考虑先按 `SK_ID_PREV` 聚合每笔贷款，再按 `SK_ID_CURR` 二次聚合

**3. 主表自身特征未充分利用**
- `application_train.csv` 原始 120 列直接使用，未做交叉特征
- 信贷领域经典指标如负债收入比（`AMT_CREDIT / AMT_INCOME_TOTAL`）、月供收入比（`AMT_ANNUITY / AMT_INCOME_TOTAL`）未提取

**4. 缺少计数类特征**
- 各子表中每人的记录数本身就是有用特征（反映信用历史丰富程度）

### 综合评分（满分 10）

| 维度 | 分数 | 说明 |
|------|------|------|
| 业务理解 | 7/10 | 特征有风险逻辑，不是机械聚合 |
| 覆盖广度 | 5/10 | 每张表都涉及，但每张表挖掘深度不足 |
| 技术实现 | 6/10 | 代码正确，但模式重复，可封装优化 |
| 竞赛竞争力 | 4/10 | 作为 baseline 可以，距银牌方案有差距 |

> **说明**：当前目标是学习数据挖掘全流程，现有方案足够跑通后续特征选择、建模、调参。走完一遍后再回来加深特征工程会更有方向感。

In [1]:
import pandas as pd
import numpy as np

DATA_DIR = '/home/ye/data-science/home-credit-default-risk/data/'
df_bureau = pd.read_csv(DATA_DIR+'bureau.csv')
app_train = pd.read_csv(DATA_DIR+'application_train.csv')
bureau_loan_count = df_bureau.groupby("SK_ID_CURR")["SK_ID_BUREAU"].count().rename("bureau_loan_count")
bureau_credit_sum_stats = df_bureau.groupby("SK_ID_CURR")["AMT_CREDIT_SUM"].agg(["mean","sum","std"])
bureau_credit_sum_stats = bureau_credit_sum_stats.rename(columns={
    "sum": "bureau_credit_sum_total",
    "mean": "bureau_credit_sum_mean",
    "std": "bureau_credit_sum_std"
})
bureau_credit_sum_stats["bureau_credit_sum_std"] = bureau_credit_sum_stats["bureau_credit_sum_std"].fillna(0)
app_train = app_train.merge(bureau_loan_count, on = "SK_ID_CURR" , how = "left" )
app_train["bureau_loan_count"] = app_train["bureau_loan_count"].fillna(0)
app_train = app_train.merge(bureau_credit_sum_stats , on= "SK_ID_CURR" ,how= "left")
app_train[["bureau_credit_sum_mean","bureau_credit_sum_total","bureau_credit_sum_std"]] = app_train[["bureau_credit_sum_mean","bureau_credit_sum_total","bureau_credit_sum_std"]].fillna(0)
bureau_overdue_stats = df_bureau.groupby("SK_ID_CURR").agg(
    {
        "CREDIT_DAY_OVERDUE":"mean",
        "AMT_CREDIT_MAX_OVERDUE" : ["max","sum"],
        "CNT_CREDIT_PROLONG" : "sum"
    }
)
bureau_overdue_stats.columns=[
    "bureau_overdue_days_mean",
    "bureau_max_overdue_max",
    "bureau_max_overdue_sum",
    "bureau_prolong_sum"
]
app_train = app_train.merge(bureau_overdue_stats, on="SK_ID_CURR",how="left")
app_train[[
    "bureau_overdue_days_mean",
    "bureau_max_overdue_max",
    "bureau_max_overdue_sum",
    "bureau_prolong_sum"
]] = app_train[[
    "bureau_overdue_days_mean",
    "bureau_max_overdue_max",
    "bureau_max_overdue_sum",
    "bureau_prolong_sum"
]].fillna(0)
temp = df_bureau[df_bureau["CREDIT_ACTIVE"] == "Active"].groupby("SK_ID_CURR")["CREDIT_ACTIVE"].count()
bureau_active_loan_ratio = (temp/bureau_loan_count).fillna(0).rename("bureau_active_loan_ratio")
app_train = app_train.merge(bureau_active_loan_ratio,on="SK_ID_CURR",how="left").fillna({"bureau_active_loan_ratio" : 0})
app_train
     

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,AMT_REQ_CREDIT_BUREAU_YEAR,bureau_loan_count,bureau_credit_sum_mean,bureau_credit_sum_total,bureau_credit_sum_std,bureau_overdue_days_mean,bureau_max_overdue_max,bureau_max_overdue_sum,bureau_prolong_sum,bureau_active_loan_ratio
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,1.0,8.0,108131.945625,865055.565,146075.557435,0.0,5043.645,8405.145,0.0,0.250000
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0.0,4.0,254350.125000,1017400.500,372269.465535,0.0,0.000,0.000,0.0,0.250000
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0.0,2.0,94518.900000,189037.800,26.728636,0.0,0.000,0.000,0.0,0.000000
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,NaN,0.0,0.000000,0.000,0.000000,0.0,0.000,0.000,0.0,0.000000
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0.0,1.0,146250.000000,146250.000,0.000000,0.0,0.000,0.000,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307506,456251,0,Cash loans,M,N,N,0,157500.0,254700.0,27558.0,...,NaN,0.0,0.000000,0.000,0.000000,0.0,0.000,0.000,0.0,0.000000
307507,456252,0,Cash loans,F,N,Y,0,72000.0,269550.0,12001.5,...,NaN,0.0,0.000000,0.000,0.000000,0.0,0.000,0.000,0.0,0.000000
307508,456253,0,Cash loans,F,N,Y,0,153000.0,677664.0,29979.0,...,1.0,4.0,990000.000000,3960000.000,853024.032487,0.0,0.000,0.000,0.0,0.500000
307509,456254,1,Cash loans,F,N,Y,0,171000.0,370107.0,20205.0,...,0.0,1.0,45000.000000,45000.000,0.000000,0.0,0.000,0.000,0.0,0.000000


## Bureau 表特征工程

从 `bureau.csv` 提取每个申请人在其他金融机构的历史信贷信息。

由于 bureau 表是一对多关系（一个申请人对应多条记录），需通过聚合将多行压缩为一行后合并回主表。

**提取的特征：**
- `bureau_loan_count`：历史贷款总次数
- `bureau_credit_sum_mean` / `total` / `std`：贷款金额的均值、总额、标准差
- `bureau_overdue_days_mean`：平均逾期天数
- `bureau_max_overdue_max` / `sum`：历史最大逾期金额的最大值和累计值
- `bureau_prolong_sum`：贷款延期总次数
- `bureau_active_loan_ratio`：活跃贷款占比（活跃数 / 总数）

In [2]:
df_prev = pd.read_csv(DATA_DIR+'previous_application.csv')
prev_refused = df_prev[df_prev["NAME_CONTRACT_STATUS"] == "Refused"].groupby("SK_ID_CURR")["NAME_CONTRACT_STATUS"].count()
prev_approved = df_prev[df_prev["NAME_CONTRACT_STATUS"] == "Approved"].groupby("SK_ID_CURR")["NAME_CONTRACT_STATUS"].count()
prev_all_count = df_prev.groupby("SK_ID_CURR")["NAME_CONTRACT_STATUS"].count()
prev_refused_ratio = (prev_refused / prev_all_count).rename("prev_refused_ratio")
prev_approved_ratio = (prev_approved /prev_all_count).rename("prev_approved_ratio")
prev_credit_fulfill_mean = (df_prev["AMT_CREDIT"]/df_prev["AMT_APPLICATION"]).groupby(df_prev["SK_ID_CURR"]).mean().rename("prev_credit_fulfill_mean")
prev_term_mean = df_prev.groupby("SK_ID_CURR")["CNT_PAYMENT"].mean().rename("prev_term_mean")
prev_term_max = df_prev.groupby("SK_ID_CURR")["CNT_PAYMENT"].max().rename("prev_term_max")
app_train = app_train.merge(prev_approved_ratio, on="SK_ID_CURR", how="left").fillna({"prev_approved_ratio" : 0})
app_train = app_train.merge(prev_refused_ratio, on="SK_ID_CURR", how="left").fillna({"prev_refused_ratio":0})
app_train = app_train.merge(prev_credit_fulfill_mean, on="SK_ID_CURR", how="left").fillna({"prev_credit_fulfill_mean":0})
app_train = app_train.merge(prev_term_mean, on="SK_ID_CURR", how="left").fillna({"prev_term_mean":0})
app_train = app_train.merge(prev_term_max, on="SK_ID_CURR", how="left").fillna({"prev_term_max":0})
app_train

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,bureau_overdue_days_mean,bureau_max_overdue_max,bureau_max_overdue_sum,bureau_prolong_sum,bureau_active_loan_ratio,prev_approved_ratio,prev_refused_ratio,prev_credit_fulfill_mean,prev_term_mean,prev_term_max
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0.0,5043.645,8405.145,0.0,0.250000,1.000000,0.000000,1.000000,24.000000,24.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0.0,0.000,0.000,0.0,0.250000,1.000000,0.000000,1.057664,10.000000,12.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0.0,0.000,0.000,0.0,0.000000,1.000000,0.000000,0.828021,4.000000,4.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0.0,0.000,0.000,0.0,0.000000,0.555556,0.111111,1.012684,23.000000,48.0
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0.0,0.000,0.000,0.0,0.000000,1.000000,0.000000,1.046356,20.666667,48.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307506,456251,0,Cash loans,M,N,N,0,157500.0,254700.0,27558.0,...,0.0,0.000,0.000,0.0,0.000000,1.000000,0.000000,1.000000,8.000000,8.0
307507,456252,0,Cash loans,F,N,Y,0,72000.0,269550.0,12001.5,...,0.0,0.000,0.000,0.0,0.000000,1.000000,0.000000,0.986561,6.000000,6.0
307508,456253,0,Cash loans,F,N,Y,0,153000.0,677664.0,29979.0,...,0.0,0.000,0.000,0.0,0.500000,1.000000,0.000000,0.831397,5.000000,6.0
307509,456254,1,Cash loans,F,N,Y,0,171000.0,370107.0,20205.0,...,0.0,0.000,0.000,0.0,0.000000,1.000000,0.000000,1.122050,15.000000,16.0


## Previous Application 表特征工程

从 `previous_application.csv` 提取每个申请人在本平台的历史贷款申请信息。

同样是一对多关系（一个申请人对应多条历史申请），需通过聚合或比例计算压缩为一行后合并回主表。

**提取的特征：**
- `prev_approved_ratio`：历史申请被批准的比例
- `prev_refused_ratio`：历史申请被拒绝的比例
- `prev_credit_fulfill_mean`：信贷满足率均值（实际批准金额 / 申请金额），反映银行对申请人的风险判断
- `prev_term_mean`：平均分期期数
- `prev_term_max`：最长分期期数

In [3]:
df_install = pd.read_csv(DATA_DIR+'installments_payments.csv')
install_days_delay_mean = (df_install["DAYS_ENTRY_PAYMENT"]-df_install["DAYS_INSTALMENT"]).groupby(df_install["SK_ID_CURR"]).mean().rename("install_days_delay_mean")
install_amt_diff_mean = (df_install["AMT_PAYMENT"]-df_install["AMT_INSTALMENT"]).groupby(df_install["SK_ID_CURR"]).mean().rename("install_amt_diff_mean")
install_days_delay_max = (df_install["DAYS_ENTRY_PAYMENT"] - df_install["DAYS_INSTALMENT"]).groupby(df_install["SK_ID_CURR"]).max().rename("install_days_delay_max")
app_train = app_train.merge(install_days_delay_mean,on="SK_ID_CURR",how="left").fillna({"install_days_delay_mean":0})
app_train = app_train.merge(install_amt_diff_mean,on="SK_ID_CURR",how="left").fillna({"install_amt_diff_mean":0})
app_train = app_train.merge(install_days_delay_max,on="SK_ID_CURR",how="left").fillna({"install_days_delay_max":0})
app_train  

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,bureau_prolong_sum,bureau_active_loan_ratio,prev_approved_ratio,prev_refused_ratio,prev_credit_fulfill_mean,prev_term_mean,prev_term_max,install_days_delay_mean,install_amt_diff_mean,install_days_delay_max
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0.0,0.250000,1.000000,0.000000,1.000000,24.000000,24.0,-20.421053,0.000000,-12.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0.0,0.250000,1.000000,0.000000,1.057664,10.000000,12.0,-7.160000,0.000000,-1.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0.0,0.000000,1.000000,0.000000,0.828021,4.000000,4.0,-7.666667,0.000000,-3.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0.0,0.000000,0.555556,0.111111,1.012684,23.000000,48.0,-19.375000,0.000000,-1.0
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0.0,0.000000,1.000000,0.000000,1.046356,20.666667,48.0,-3.636364,-452.384318,12.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307506,456251,0,Cash loans,M,N,N,0,157500.0,254700.0,27558.0,...,0.0,0.000000,1.000000,0.000000,1.000000,8.000000,8.0,-36.285714,0.000000,-8.0
307507,456252,0,Cash loans,F,N,Y,0,72000.0,269550.0,12001.5,...,0.0,0.000000,1.000000,0.000000,0.986561,6.000000,6.0,-2.833333,0.000000,3.0
307508,456253,0,Cash loans,F,N,Y,0,153000.0,677664.0,29979.0,...,0.0,0.500000,1.000000,0.000000,0.831397,5.000000,6.0,-14.500000,-283.792500,9.0
307509,456254,1,Cash loans,F,N,Y,0,171000.0,370107.0,20205.0,...,0.0,0.000000,1.000000,0.000000,1.122050,15.000000,16.0,-19.000000,0.000000,-8.0


## Installments Payments 表特征工程

从 `installments_payments.csv` 提取每个申请人的分期还款行为信息。

每条记录是一次分期还款，包含应还日期/金额和实还日期/金额，通过计算差值可以衡量还款纪律。

**提取的特征：**
- `install_days_delay_mean`：平均还款延迟天数（负数=提前还款）
- `install_amt_diff_mean`：平均还款金额差异（负数=少还）
- `install_days_delay_max`：最大还款延迟天数（反映极端拖欠行为）

In [4]:
df_pos = pd.read_csv(DATA_DIR+'POS_CASH_balance.csv')
pos_dpd_mean = df_pos.groupby("SK_ID_CURR")["SK_DPD"].mean().rename("pos_dpd_mean")
pos_dpd_max = df_pos.groupby("SK_ID_CURR")["SK_DPD"].max().rename("pos_dpd_max")
pos_dpd_over0_ratio = (df_pos[df_pos["SK_DPD"] > 0].groupby("SK_ID_CURR").size() / df_pos.groupby("SK_ID_CURR").size()).rename("pos_dpd_over0_ratio")
pos_demand_ratio = (df_pos[(df_pos["NAME_CONTRACT_STATUS"] == "Demand") | (df_pos["NAME_CONTRACT_STATUS"] == "Amortized debt")].groupby("SK_ID_CURR").size() / df_pos.groupby("SK_ID_CURR").size()).rename("pos_demand_ratio")
pos_completed_ratio = (df_pos[df_pos["NAME_CONTRACT_STATUS"] == "Completed"].groupby("SK_ID_CURR").size() / df_pos.groupby("SK_ID_CURR").size()).rename("pos_completed_ratio")
app_train = app_train.merge(pos_dpd_mean,on="SK_ID_CURR",how="left").fillna({"pos_dpd_mean":0})
app_train = app_train.merge(pos_dpd_max,on="SK_ID_CURR",how="left").fillna({"pos_dpd_max":0})
app_train = app_train.merge(pos_dpd_over0_ratio,on="SK_ID_CURR",how="left").fillna({"pos_dpd_over0_ratio":0})
app_train = app_train.merge(pos_demand_ratio,on="SK_ID_CURR",how="left").fillna({"pos_demand_ratio":0})
app_train = app_train.merge(pos_completed_ratio,on="SK_ID_CURR",how="left").fillna({"pos_completed_ratio":0})
app_train
 

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,prev_term_mean,prev_term_max,install_days_delay_mean,install_amt_diff_mean,install_days_delay_max,pos_dpd_mean,pos_dpd_max,pos_dpd_over0_ratio,pos_demand_ratio,pos_completed_ratio
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,24.000000,24.0,-20.421053,0.000000,-12.0,0.000000,0.0,0.000000,0.0,0.000000
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,10.000000,12.0,-7.160000,0.000000,-1.0,0.000000,0.0,0.000000,0.0,0.071429
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,4.000000,4.0,-7.666667,0.000000,-3.0,0.000000,0.0,0.000000,0.0,0.250000
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,23.000000,48.0,-19.375000,0.000000,-1.0,0.000000,0.0,0.000000,0.0,0.095238
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,20.666667,48.0,-3.636364,-452.384318,12.0,0.000000,0.0,0.000000,0.0,0.045455
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307506,456251,0,Cash loans,M,N,N,0,157500.0,254700.0,27558.0,...,8.000000,8.0,-36.285714,0.000000,-8.0,0.000000,0.0,0.000000,0.0,0.111111
307507,456252,0,Cash loans,F,N,Y,0,72000.0,269550.0,12001.5,...,6.000000,6.0,-2.833333,0.000000,3.0,0.000000,0.0,0.000000,0.0,0.142857
307508,456253,0,Cash loans,F,N,Y,0,153000.0,677664.0,29979.0,...,5.000000,6.0,-14.500000,-283.792500,9.0,0.294118,5.0,0.058824,0.0,0.117647
307509,456254,1,Cash loans,F,N,Y,0,171000.0,370107.0,20205.0,...,15.000000,16.0,-19.000000,0.000000,-8.0,0.000000,0.0,0.000000,0.0,0.000000


## POS_CASH_balance 表特征工程

从 `POS_CASH_balance.csv` 提取每个申请人在本平台 POS/现金贷款的月度还款快照信息。

每条记录是某笔贷款某个月的状态快照（一对多关系），通过聚合提取逾期行为和合同状态特征。

**提取的特征：**
- `pos_dpd_mean`：月均逾期天数（SK_DPD 均值）
- `pos_dpd_max`：最大单月逾期天数（反映极端逾期）
- `pos_dpd_over0_ratio`：逾期月份占比（SK_DPD > 0 的月份数 / 总月份数）
- `pos_demand_ratio`：催收/坏账状态占比（Demand + Amortized debt 记录数 / 总记录数）
- `pos_completed_ratio`：已完成状态占比（Completed 记录数 / 总记录数）

In [5]:
df_credit = pd.read_csv(DATA_DIR+'credit_card_balance.csv')
cc_utilization_mean = (df_credit["AMT_BALANCE"] / df_credit["AMT_CREDIT_LIMIT_ACTUAL"].replace(0, np.nan)).groupby(df_credit["SK_ID_CURR"]).mean().rename("cc_utilization_mean")
cc_atm_drawing_ratio_mean = (df_credit["AMT_DRAWINGS_ATM_CURRENT"] / df_credit["AMT_DRAWINGS_CURRENT"].replace(0, np.nan)).groupby(df_credit["SK_ID_CURR"]).mean().rename("cc_atm_drawing_ratio_mean")
cc_min_payment_ratio = (df_credit[df_credit["AMT_PAYMENT_CURRENT"] <= df_credit["AMT_INST_MIN_REGULARITY"]* 1.05].groupby("SK_ID_CURR").size() / df_credit.groupby("SK_ID_CURR").size()).rename("cc_min_payment_ratio")
cc_dpd_mean = df_credit.groupby("SK_ID_CURR")["SK_DPD"].mean().rename("cc_dpd_mean")
cc_dpd_max = df_credit.groupby("SK_ID_CURR")["SK_DPD"].max().rename("cc_dpd_max")
app_train = app_train.merge(cc_utilization_mean,on="SK_ID_CURR",how="left").fillna({"cc_utilization_mean":0})
app_train = app_train.merge(cc_atm_drawing_ratio_mean,on="SK_ID_CURR",how="left").fillna({"cc_atm_drawing_ratio_mean":0})
app_train = app_train.merge(cc_min_payment_ratio,on="SK_ID_CURR",how="left").fillna({"cc_min_payment_ratio":0})
app_train = app_train.merge(cc_dpd_mean,on="SK_ID_CURR",how="left").fillna({"cc_dpd_mean":0})
app_train = app_train.merge(cc_dpd_max,on="SK_ID_CURR",how="left").fillna({"cc_dpd_max":0})
print(app_train[["cc_utilization_mean", "cc_atm_drawing_ratio_mean", "cc_min_payment_ratio", "cc_dpd_mean", "cc_dpd_max"]].describe())

       cc_utilization_mean  cc_atm_drawing_ratio_mean  cc_min_payment_ratio  \
count        307511.000000              307511.000000         307511.000000   
mean              0.091483                   0.122938              0.061381   
std               0.226203                   0.302308              0.168680   
min              -0.084848                   0.000000              0.000000   
25%               0.000000                   0.000000              0.000000   
50%               0.000000                   0.000000              0.000000   
75%               0.000000                   0.000000              0.000000   
max               2.138790                   1.000000              1.000000   

         cc_dpd_mean     cc_dpd_max  
count  307511.000000  307511.000000  
mean        1.189522       4.791191  
std        23.786480      76.702010  
min         0.000000       0.000000  
25%         0.000000       0.000000  
50%         0.000000       0.000000  
75%         0.000000  

## Credit Card Balance 表特征工程

从 `credit_card_balance.csv` 提取每个申请人在本平台信用卡的月度使用和还款信息。

每条记录是某张信用卡某个月的快照（一对多关系），通过聚合提取额度使用、消费行为和逾期特征。

**提取的特征：**
- `cc_utilization_mean`：平均信用卡额度使用率（余额 / 额度，越高说明资金越紧张）
- `cc_atm_drawing_ratio_mean`：ATM 取现占总消费的比例（取现多暗示现金流紧张）
- `cc_min_payment_ratio`：只还最低还款额的月份占比（只还最低说明还款能力不足）
- `cc_dpd_mean`：月均逾期天数
- `cc_dpd_max`：最大单月逾期天数

In [6]:
app_train.to_csv("../data/processed/app_train_features.csv", index=False)
print(f"已保存，shape: {app_train.shape}")

已保存，shape: (307511, 149)
